# Taxi Trip Analysis — Automated EDA

This notebook loads Seaborn's `taxis` dataset, samples 3,000 rows with a fixed seed, creates ydata-profiling and Sweetviz reports, launches D-Tale for interactive exploration, and verifies the strongest numeric correlation in code.

**Install dependencies** in the active notebook environment before running:
```bash
python -m pip install seaborn ydata-profiling sweetviz dtale matplotlib pandas
```


In [ ]:
import itertools
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from ydata_profiling import ProfileReport
import sweetviz as sv
import dtale

# Project folders (the notebook is expected to run from the repository root)
ROOT = Path.cwd()
NOTEBOOK_DIR = ROOT / "notebooks"
REPORTS_DIR = ROOT / "reports"
PROCESSED_DIR = ROOT / "data" / "processed"

for folder in (REPORTS_DIR, PROCESSED_DIR):
    folder.mkdir(parents=True, exist_ok=True)


## 1. Load and sample the Seaborn taxis dataset

In [ ]:
taxi = sns.load_dataset("taxis")

taxi_sample = taxi.sample(n=3000, random_state=42).copy()
print("Sample shape:", taxi_sample.shape)

# Keep a reproducible copy of the exact sample used for analysis.
taxi_sample.to_csv(PROCESSED_DIR / "taxi_sample.csv", index=False)
taxi_sample.head()


In [ ]:
profile = ProfileReport(
    taxi_sample,
    title="Taxi Trips Profile",
    explorative=True
)
profile.to_file(REPORTS_DIR / "taxi_profile.html")
print("Saved:", REPORTS_DIR / "taxi_profile.html")


## 2. Sweetviz: cash trips versus all other payment types

In [ ]:
cash_mask = taxi_sample["payment"].eq("cash")

sweetviz_report = sv.compare_intra(
    taxi_sample,
    cash_mask,
    ["Cash", "Other"],
    target_feat="fare"
)
sweetviz_report.show_html(
    filepath=str(REPORTS_DIR / "taxi_sweetviz_cash_vs_other.html"),
    open_browser=False
)
print("Cash rows:", int(cash_mask.sum()))
print("Other rows:", int((~cash_mask).sum()))


## 3. D-Tale interactive exploration

Run this cell, then use the printed URL to open D-Tale. In the D-Tale interface, open **Correlations** and select `distance`, `fare`, `tip`, and `total`; then open **Describe** for `tip`.

Save a screenshot of the correlation heatmap to `reports/dtale_correlations.png`. The screenshot is an interactive-browser capture, so complete that step manually after opening the D-Tale page.


In [ ]:
d = dtale.show(taxi_sample)
print("D-Tale URL:", d.main_url())
# If the browser does not open automatically, uncomment:
# d.open_browser()


### D-Tale screenshot checklist

1. Open the D-Tale URL printed above.
2. Open **Correlations** and choose `distance`, `fare`, `tip`, and `total`.
3. Open **Describe** and select `tip`.
4. Capture the correlation heatmap and save it as `reports/dtale_correlations.png`.


## 4. Correlation verification and Seaborn heatmap

In [ ]:
numeric_corr = taxi_sample.corr(numeric_only=True)

# Find the strongest absolute correlation while excluding the diagonal
# and every pair that contains the 'total' column.
valid_pairs = []
for col_a, col_b in itertools.combinations(numeric_corr.columns, 2):
    if "total" in (col_a, col_b):
        continue
    value = numeric_corr.loc[col_a, col_b]
    if pd.notna(value):
        valid_pairs.append((col_a, col_b, value))

if not valid_pairs:
    raise ValueError("No eligible numeric column pairs were available.")

strongest_pair = max(valid_pairs, key=lambda item: abs(item[2]))
print(
    "Strongest eligible relationship:",
    f"{strongest_pair[0]} and {strongest_pair[1]}",
    f"(correlation = {strongest_pair[2]:.4f})"
)

plt.figure(figsize=(12, 9))
sns.heatmap(
    numeric_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Taxi Sample — Numeric Correlation Heatmap")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "taxi_corr_check.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", REPORTS_DIR / "taxi_corr_check.png")


## Final output checklist

After **Restart Kernel and Run All**, confirm that these files exist:

- `reports/taxi_profile.html`
- `reports/taxi_sweetviz_cash_vs_other.html`
- `reports/taxi_corr_check.png`
- `reports/dtale_correlations.png` (captured manually from the D-Tale browser)
- `data/processed/taxi_sample.csv`

The first four are the requested reports and images; the processed CSV preserves the sampled data used to produce them.
